# Neutron–Xenon Scattering Explorer

**Project:** `neutron-xenon-scattering` · **Edition:** Colab.

This notebook and the static website use **one identical data bundle and the same browser rendering code**. It contains ENDF/B-VII.1 and the qualified JENDL-4.0u comparison, the spherical optical grid, saved Xe-132 OPTMAN examples, and native DD dip/area diagnostics. Both libraries start selected.

**Run the setup and display cells below.** `BASE_URL` points to the website's shared data package. The notebook downloads and verifies that package; Google Drive authorization is unnecessary. For an offline copy, clear `BASE_URL` and use a full `PACKAGE_DIR` or `PACKAGE_ZIP` path. All four settings blank prompts for a ZIP upload in Colab. No raw evaluation files, package installation, Fortran compilation, or NERSC access are needed.


The notebook is saved with outputs cleared to stay small. Colab runtimes are temporary; rerun setup after a restart. Saving the rendered output will enlarge the notebook. A modern browser with JavaScript and gzip DecompressionStream support is required. If local Jupyter suppresses scripts, trust the notebook after reviewing its code and run the display cell again.

## Cross-section accounting

For the spherical teaching model, with energy-averaged shape and compound components:

$$\sigma_{\mathrm{total}}=\sigma_{\mathrm{elastic}}+\sigma_{\mathrm{nonelastic}}$$
$$\sigma_{\mathrm{elastic}}=\sigma_{\mathrm{shape}}+\sigma_{\mathrm{compound\ elastic}}$$
$$\sigma_{\mathrm{optical\ reaction}}=\sigma_{\mathrm{compound\ elastic}}+\sigma_{\mathrm{nonelastic}}$$

The slider assigns $\sigma_{\mathrm{compound\ elastic}}=f\sigma_{\mathrm{optical\ reaction}}$. Total stays fixed; elastic rises and non-elastic falls. Evaluated curves are unchanged. The website below contains the full beginner explanation and plot guide. The OPTMAN coupled-channel panel uses **absorption after explicit direct channels** as its denominator. No POD prediction is included.

## Is ENDF minus JENDL approximately constant?

The fifth interactive plot compares the two evaluated elastic predictions directly, at the same neutron energy, target and CM angle:

$$\Delta(\theta)=\left(\frac{d\sigma}{d\Omega}\right)_{\mathrm{ENDF/B\text{-}VII.1}}-\left(\frac{d\sigma}{d\Omega}\right)_{\mathrm{JENDL\text{-}4.0u}}.$$
$$C=\frac{\sigma_{\mathrm{el,ENDF}}-\sigma_{\mathrm{el,JENDL}}}{4\pi}=\frac{1}{4\pi}\int\Delta(\theta)\,d\Omega,\qquad
\mathrm{RMS}=\sqrt{\frac{1}{4\pi}\int[\Delta(\theta)-C]^2\,d\Omega}.$$

Purple is the signed difference in b/sr; the black dashed line is the isotropic difference with the same integrated elastic area. Solid-angle weighting uses $d\Omega=2\pi\sin\theta\,d\theta$, not equal weight per degree. The RMS is descriptive, not a fit uncertainty. These are absolute differential cross sections, not normalized angular probabilities.

A nearly flat curve supports an approximately isotropic difference; it does not identify compound elastic. A sign change rules out adding a positive isotropic term alone as the explanation over all angles. No optical model is subtracted. The plot follows energy and target but is independent of the compound slider and library overlay checkboxes. Its default fixed y range is −0.25 to +0.25 b/sr; use the manual difference-range selector to zoom or inspect clipped values. Unsupported resonance ranges stay masked.


In [ ]:
# Website edition: both interfaces load the same public data bundle.
BASE_URL = "https://sinochen1992.github.io/neutron-xenon-scattering/"
DRIVE_FILE = ""     # Optional alternate restricted Drive file; clear BASE_URL first.
PACKAGE_ZIP = ""    # Optional full path to a local package ZIP.
PACKAGE_DIR = ""    # Optional full path to the unpacked package.


In [ ]:
"""Portable setup; restricted Drive downloads use Colab's built-in authentication."""
import base64
import gzip
import hashlib
import io
import json
from pathlib import Path
import re
import urllib.parse
import urllib.request
import uuid
import zipfile

ASSET_NAMES = (
    'assets/data.json.gz', 'assets/panels.json', 'assets/app.js',
    'assets/style.css', 'index.html', 'neutron_optical_model.py',
)
MAX_PACKAGE_BYTES = 30_000_000


def drive_file_id(link):
    """Accept a Drive file sharing URL or file ID; reject folders/other hosts."""
    value = link.strip()
    if re.fullmatch(r'[A-Za-z0-9_-]{10,200}', value):
        return value
    url = urllib.parse.urlparse(value)
    if url.scheme != 'https' or url.netloc != 'drive.google.com':
        raise ValueError('DRIVE_FILE must be a Google Drive file sharing URL or file ID.')
    match = re.fullmatch(r'/file/d/([A-Za-z0-9_-]{10,200})(?:/view)?/?', url.path)
    file_id = match[1] if match else ''
    if url.path in ('/open', '/uc'):
        file_id = urllib.parse.parse_qs(url.query).get('id', [''])[0]
    if not re.fullmatch(r'[A-Za-z0-9_-]{10,200}', file_id):
        raise ValueError('Use the ZIP file link, not a Drive folder or shortcut link.')
    return file_id


def download_drive_zip(link):
    """Read exactly the configured Drive file using the notebook user's access.

    Authentication runs only when this function is called inside Colab. No
    Drive mount, file listing, permission change, or anonymous fallback occurs.
    Official references: Google Colab auth.authenticate_user; Drive API v3
    files.get with alt=media (developers.google.com/workspace/drive/api/guides/manage-downloads).
    """
    file_id = drive_file_id(link)
    try:
        from google.colab import auth
        import google.auth
        from google.auth.transport.requests import AuthorizedSession
    except ImportError as exc:
        raise RuntimeError('Restricted Drive loading needs Google Colab. In local Jupyter, download the ZIP with your browser, clear DRIVE_FILE, and set PACKAGE_ZIP.') from exc
    print('Sign in to Colab authorization with a Google account that can download this ZIP.')
    auth.authenticate_user()
    credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
    endpoint = 'https://www.googleapis.com/drive/v3/files/' + file_id

    def check_response(response):
        if response.status_code in (401, 403, 404):
            raise PermissionError('Google Drive refused access (HTTP ' + str(response.status_code) + '). Check the selected account, file sharing/download permissions, and Colab Drive authorization. The ZIP can remain restricted.')
        if response.status_code != 200:
            raise RuntimeError('Google Drive download failed (HTTP ' + str(response.status_code) + '). Retry later or download the ZIP with your browser and use PACKAGE_ZIP.')

    with AuthorizedSession(credentials) as session:
        # Check the exact file's download capability before requesting its data.
        with session.get(endpoint, params={'fields': 'id,mimeType,size,capabilities(canDownload)',
                                          'supportsAllDrives': 'true'}, timeout=40) as response:
            check_response(response)
            metadata = response.json()
        if metadata.get('capabilities', {}).get('canDownload') is not True:
            raise PermissionError('This account cannot download the selected Drive file. Ask the owner to allow downloads for that account; public sharing is unnecessary.')
        if metadata.get('mimeType', '').startswith('application/vnd.google-apps.'):
            raise ValueError('DRIVE_FILE must identify the ZIP itself, not a folder, shortcut, or Google document.')
        if int(metadata.get('size', 0)) > MAX_PACKAGE_BYTES:
            raise ValueError('Drive file exceeds the portable package size limit.')
        with session.get(endpoint, params={'alt': 'media', 'supportsAllDrives': 'true'},
                         stream=True, timeout=40) as response:
            check_response(response)
            content = bytearray()
            for chunk in response.iter_content(chunk_size=1024*1024):
                content.extend(chunk)
                if len(content) > MAX_PACKAGE_BYTES:
                    raise ValueError('Drive file exceeds the portable package size limit.')
    content = bytes(content)
    if not zipfile.is_zipfile(io.BytesIO(content)):
        raise ValueError('Google Drive returned a file that is not a ZIP package.')
    return content


def load_package(*, base_url='', archive='', directory='', drive_file=''):
    """Read one shared package from a folder, ZIP, site, restricted Drive, or upload."""
    if sum(bool(x) for x in (base_url, archive, directory, drive_file)) > 1:
        raise ValueError('Choose only one package source.')
    z = None
    if directory:
        root = Path(directory).expanduser().resolve()
        read = lambda name: (root / name).read_bytes()
    elif base_url:
        if not base_url.startswith(('https://', 'http://localhost:', 'http://127.0.0.1:')):
            raise ValueError('Use an HTTPS site URL (or localhost for testing).')
        def read(name):
            with urllib.request.urlopen(base_url.rstrip('/') + '/' + name, timeout=40) as response:
                result = response.read(30_000_001)
            if len(result) > 30_000_000:
                raise ValueError('Asset exceeds the portable package size limit.')
            return result
    else:
        if drive_file:
            source = download_drive_zip(drive_file)
        elif archive:
            source = Path(archive).expanduser().read_bytes()
        else:
            try:
                from google.colab import files
            except ImportError as exc:
                raise RuntimeError('Set PACKAGE_DIR or PACKAGE_ZIP, or use Colab to upload the package ZIP.') from exc
            print('Upload neutron_xe_portable.zip from the portable package.')
            uploaded = files.upload()
            if len(uploaded) != 1:
                raise ValueError('Upload exactly one portable ZIP.')
            source = next(iter(uploaded.values()))
        z = zipfile.ZipFile(io.BytesIO(source))
        # Read an explicit allowlist without extracting paths to the filesystem.
        def read(name):
            if z.getinfo(name).file_size > 30_000_000:
                raise ValueError('Asset exceeds the portable package size limit.')
            return z.read(name)
    try:
        manifest_bytes = read('manifest.json')
        manifest = json.loads(manifest_bytes)
        if manifest.get('schema') != 1 or set(manifest.get('files', {})) != set(ASSET_NAMES):
            raise ValueError('Unexpected portable manifest schema or asset list.')
        assets = {'manifest.json': manifest_bytes}
        for name in ASSET_NAMES:
            content = read(name)
            expected = manifest['files'][name]
            if len(content) != expected['bytes'] or hashlib.sha256(content).hexdigest() != expected['sha256']:
                raise ValueError('Shared package checksum mismatch: ' + name)
            assets[name] = content
    finally:
        if z is not None:
            z.close()
    bundle = json.loads(gzip.decompress(assets['assets/data.json.gz']))
    if bundle.get('schema') != 1 or set(bundle['main']['libraries']) != {'endfb71', 'jendl40'}:
        raise ValueError('Unexpected data schema or libraries.')
    return assets, bundle


def explorer_html(assets):
    """Embed the identical static app in an isolated, auto-sized notebook frame."""
    encoded = {name: base64.b64encode(assets[name]).decode('ascii') for name in
               ('manifest.json', 'assets/data.json.gz', 'assets/panels.json')}
    document = assets['index.html'].decode('utf-8')
    # Keep the selected display name when loading an earlier compatible ZIP.
    document = document.replace('Neutron–xenon scattering explorer', 'Neutron–Xenon Scattering Explorer')
    document = document.replace('<h1>Neutron–xenon scattering</h1>', '<h1>Neutron–Xenon Scattering Explorer</h1>')
    document = document.replace('<link rel="stylesheet" href="assets/style.css">',
                                '<style>' + assets['assets/style.css'].decode('utf-8') + '</style>')
    embedded = json.dumps({'files': encoded}, separators=(',', ':')).replace('</', '<\\/')
    script = '<script>window.NXE_EMBEDDED=' + embedded + ';</script>\n<script>' + assets['assets/app.js'].decode('utf-8') + '</script>'
    document = document.replace('<script src="assets/app.js"></script>', script)
    frame_id = 'nxe-colab-' + uuid.uuid4().hex
    # Listener is registered before srcdoc is attached. Messages are source-checked.
    return ('<iframe id="' + frame_id + '" title="Neutron–Xenon Scattering Explorer" '
            'style="width:100%;height:1200px;border:0"></iframe><script>(function(){'
            'var f=document.getElementById(' + json.dumps(frame_id) + ');'
            'window.addEventListener("message",function(e){'
            'if(e.source===f.contentWindow&&e.data&&e.data.type==="nxe-page-height"&&Number.isFinite(e.data.height))'
            '{f.style.height=Math.max(500,Math.min(30000,e.data.height+8))+"px";'
            'if(window.google&&google.colab&&google.colab.output)google.colab.output.setIframeHeight(document.documentElement.scrollHeight,true);}});'
            'f.srcdoc=' + json.dumps(document).replace('</', '<\\/') + ';})();</script>')

ASSETS, BUNDLE = load_package(base_url=BASE_URL, archive=PACKAGE_ZIP, directory=PACKAGE_DIR,
                             drive_file=DRIVE_FILE)
if 'function libraryDifference(' not in json.loads(ASSETS['assets/panels.json'])['main']:
    raise ValueError("This package predates the ENDF-minus-JENDL plot. Replace the ZIP at your selected source with the current neutron_xe_portable.zip, or choose the current local/site package.")
size = len(ASSETS['assets/data.json.gz']) / 1e6
print(f"Verified shared data: {size:.2f} MB; ENDF/B-VII.1 + JENDL-4.0u.")


In [ ]:
from IPython.display import HTML, display
display(HTML(explorer_html(ASSETS)))

## Access the shared arrays

The following cells are optional. Float32 display arrays reproduce the browser grid; masks and supported bounds are stored beside them. Native precision work requires the original evaluation files and reader, which are outside this lightweight package.

In [ ]:
# Arrays are available for your own calculations without reading any raw ENDF files.
import numpy as np

def grid_array(field, library=None):
    d = BUNDLE['main'] if library is None else BUNDLE['main']['libraries'][library]
    a = np.frombuffer(base64.b64decode(d[field]), dtype='<f4')
    main = BUNDLE['main']
    nt = main['nT'] if library is None else len(d['T'])
    shape = (nt, main['nA'])
    if field == 'dsdo':
        shape += (main['nTh'],)
    elif field in ('el_l', 're_l'):
        shape += (main['nL'],)
    elif field in ('T', 'th'):
        return a
    return a.reshape(shape)

print("Angular grid dimensions (energy, isotope, angle):",
      grid_array('dsdo', 'endfb71').shape)
print("Provenance and saved native diagnostic: BUNDLE['provenance'], BUNDLE['diagnostic']")


## Direct spherical-model check

This checks the saved DD baseline with the original Python solver. It does not rerun OPTMAN or reconstruct native evaluated angular distributions.

In [ ]:
# Optional: run a fresh spherical calculation using the bundled original model.
# NumPy and SciPy are normally already available in Colab; this cell installs nothing.
import types
model = types.ModuleType('portable_neutron_optical_model')
exec(compile(ASSETS['neutron_optical_model.py'], 'neutron_optical_model.py', 'exec'), model.__dict__)
weights = model.XE_NATURAL
cross = {a: model.cross_sections(model.s_matrix(2.45, a)) for a in weights}
shape = sum(weights[a] * cross[a]['el'] for a in weights) / sum(weights.values())
reaction = sum(weights[a] * cross[a]['re'] for a in weights) / sum(weights.values())
saved = BUNDLE['diagnostic']['summary']
np.testing.assert_allclose([shape, reaction], [saved['shape_b'], saved['optical_reaction_b']], rtol=2e-8)
print(f"Direct spherical calculation: shape = {shape:.6f} b, optical reaction = {reaction:.6f} b")
print(f"Total = {shape + reaction:.6f} b; matches the saved native diagnostic.")
for row in saved['rows']:
    f_area = (row['elastic_b'] - shape) / reaction
    print(f"{row['label']}: area-match fraction = {f_area:.6f}; "
          f"at shape minimum = {row['at_shape_dip_fraction']:.6f}")
